# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mukeshburdak/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### Ranking / Scoring

Content refresh prioritization is fundamentally a **ranking problem**. The decision is **which pages should I refresh first?** — not a yes/no outcome, but an ordering. An editor with limited time and budget needs a priority score that ranks all pages by their likelihood of benefit from a refresh. This allows them to work top-down through the ranked list, refreshing the highest-impact pages first. The output is a ranked queue, not a classification.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Verify that our task is about ranking by checking the label distribution
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("../../../data/raw/content_refresh_anonymized.csv")

# Show the task type by counting declining pages
print("Content distribution by trend (the basis for our ranking):")
print(df['trend_direction'].value_counts())
print(f"\nTotal declining pages (down): {(df['trend_direction'] == 'down').sum()}")
print(f"Percentage that are declining: {(df['trend_direction'] == 'down').mean()*100:.1f}%")
print("\n→ These declining pages are the BEST candidates for refresh. Our task is to RANK all pages by likelihood of decline.")


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Target: is_declining_label (derived, but from observed trend)

We predict **binary classification: 1 if a page is in decline, 0 otherwise**. Despite being derived, this is grounded in **observed outcomes**:
- `trend_direction` is computed from the **observed 30-day comparison**: `(impressions_last_30d - impressions_prev_30d) / impressions_prev_30d`
- Pages with `trend_direction == "down"` represent a **real, measured decline** in search impressions (> -20% drop)
- These pages **actually lost traffic** and are the best candidates for refresh intervention

This is not a rule we invented (like "low CTR"). It is the outcome pages experience — decline is observed and measured from Google Search Console data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Verify the target is grounded in observed trend metrics

# Check that is_declining_label exists or needs to be created
if 'is_declining_label' not in df.columns:
    df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Verify the link between trend_direction and our target
print("Cross-tab: trend_direction vs is_declining_label")
print(pd.crosstab(df['trend_direction'], df['is_declining_label']))

# Show trend magnitude for declining pages
print("\nTrend percentages for declining pages (should be < -20%):")
declining = df[df['trend_direction'] == 'down']
print(f"Min trend_pct: {declining['trend_pct'].min():.1f}%")
print(f"Mean trend_pct: {declining['trend_pct'].mean():.1f}%")
print(f"Max trend_pct: {declining['trend_pct'].max():.1f}%")
print("\n→ Target is OBSERVED: these pages measured real decline in impressions (GSC data), not a rule we made up.")


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Success Metric: Precision@K (top-decile precision)

**precision@10%** — of the top 10% of pages we rank as "highest priority for refresh", what fraction are actually declining?

**Why this metric:**
- **Practical constraint**: Editors can only refresh a fraction of pages (e.g., top 10% = 3,000 out of 30,000 pages)
- **Cost-benefit aligned**: precision ensures most refreshes target real decline, not wasted effort
- **Better than recall**: it's okay if we miss some declining pages (recall); it's NOT okay to tell an editor to refresh 100 pages that aren't actually declining
- **Defensible bar**: precision@10% ≥ 0.65 means 65%+ of top-ranked pages are declining. That's better than random baseline (base rate = 54.2% declining), and good enough for editorial ROI

**Baseline**: Random ranking would have ~54% precision (the base rate of decline). We aim for >60%.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Compute the base rate and define our success threshold

# Base rate of decline
base_rate = df['is_declining_label'].mean()
print(f"Base rate (% declining in dataset): {base_rate*100:.1f}%")

# Define what "top 10%" means in our dataset
top_k_count = int(len(df) * 0.10)
print(f"\nTop 10% of {len(df)} pages = {top_k_count} pages")

# If we ranked randomly, what precision would we get?
random_precision = base_rate
print(f"Random ranking precision@10%: {random_precision*100:.1f}%")

# Our success bar
success_precision = 0.65
improvement_over_random = (success_precision - random_precision) / random_precision
print(f"\nOur target: precision@10% ≥ {success_precision*100:.0f}%")
print(f"That's {improvement_over_random*100:.0f}% better than random ranking.")


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### Unit of Analysis: One row = one pseudonymized content page (article)

Each row is a single piece of content (article, comparison page, feedly item) from one of 32 client websites. The data is aggregated over a **trailing 90-day window**, giving us:
- 30,000 unique pages across 32 clients
- Content metadata (type, intent, word count, keyword context)
- 90-day traffic metrics (impressions, clicks, sessions, engagement)
- A trend comparison (last 30 days vs prior 30 days)
- The label: whether that page is declining

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Display the unit of analysis: one row = one content page

import pandas as pd

df = pd.read_csv("../../../data/raw/content_refresh_anonymized.csv")
if 'is_declining_label' not in df.columns:
    df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Show the shape and key columns
print(f"Unit of analysis: {len(df)} pages × {len(df.columns)} measured attributes")
print(f"Number of unique clients: {df['client_id'].nunique()}")
print(f"Number of unique content items: {df['content_id'].nunique()}")

# Show a few real examples (with IDs obscured)
key_cols = ['content_type', 'main_intent', 'word_count', 'impressions_90d', 'trend_direction', 'is_declining_label']
print("\nSample of 5 real pages (one row = one page):")
print(df[key_cols].head(10).to_string(index=False))

# Show label distribution
print(f"\nLabel distribution:")
print(f"  Declining (1): {(df['is_declining_label']==1).sum():,} pages")
print(f"  Not declining (0): {(df['is_declining_label']==0).sum():,} pages")
print(f"  Class balance: {(df['is_declining_label']==1).mean()*100:.1f}% declining (slightly imbalanced)")


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why ML is Necessary

A **single if-statement cannot capture which pages benefit from refresh** because the pattern is high-dimensional and non-linear:

1. **Multiple signals interact.** A page can be declining due to:
   - Low traffic + poor position (keyword opportunity not being captured)
   - High traffic + sudden drop (content became stale / outdated)
   - Medium traffic + low engagement (content isn't satisfying user intent)
   
   A rule like `if avg_position > 15 AND impressions_90d < 500 then decline` will miss pages that don't fit both conditions but are still good refresh targets.

2. **Content type changes the pattern.** A "feedly article" that has no keyword data will look declining by one signal but successful by traffic metrics — the rule changes by content type, making manual rules explode in complexity.

3. **The relationship is non-linear.** Trend magnitude alone doesn't predict refresh ROI — a page with -25% trend but 50k impressions is more valuable to refresh than a page with -30% trend but 5 impressions. No if-then captures this tradeoff elegantly.

4. **Signals are noisy.** Seasonal patterns, reporting delays, and client-specific behavior mean a fixed threshold (e.g., "rank pages by trend_pct < -20%") is brittle and doesn't learn client-specific decline patterns.

ML learns a weighted combination of signals (impressions, trend, position, engagement, content type) that generalizes better than any single person could write by hand. It also **learns per-client** if we stratify correctly.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Show why a fixed rule fails: the overlap and exceptions

import pandas as pd
import numpy as np

df = pd.read_csv("../../../data/raw/content_refresh_anonymized.csv")
if 'is_declining_label' not in df.columns:
    df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print("=== Why a fixed rule breaks ===")
print()

# Rule 1: Simple trend threshold
simple_rule = (df['trend_pct'] < -20).fillna(False)
accuracy_simple = (simple_rule.astype(int) == df['is_declining_label']).mean()
print(f"Rule: 'if trend_pct < -20%' → accuracy: {accuracy_simple*100:.1f}%")
print(f"  Precision: {df[simple_rule]['is_declining_label'].mean()*100:.1f}% (among those flagged by rule)")
print()

# Rule 2: Combine trend + position
rule_2 = (df['trend_pct'] < -20) & (df['avg_position'] > 15)
accuracy_2 = (rule_2.astype(int) == df['is_declining_label']).mean()
print(f"Rule: 'if trend_pct < -20% AND avg_position > 15' → accuracy: {accuracy_2*100:.1f}%")
print(f"  But misses {((df['is_declining_label']==1) & ~rule_2).sum()} declining pages that don't fit both conditions.")
print()

# Show interaction: high traffic + low position can still decline
print("Declining pages by traffic level:")
for tier in ['low', 'moderate', 'good', 'excellent']:
    mask = df['impression_tier'] == tier
    if mask.sum() > 0:
        pct_declining = df[mask]['is_declining_label'].mean()
        print(f"  {tier}: {pct_declining*100:.1f}% declining")

print("\n→ Pattern is too tangled. ML will learn the weighted combination.")


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.